In [27]:
!pip install openpyxl


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [28]:
from data_loaders.nutrition_table_incap_data_loader import Nutrition_INCAP
incap_data = Nutrition_INCAP("./data/raw/tabladecomposiciondealimentos.pdf")

Loaded data from cache serialized data


In [29]:
# Load data from 2024 - 2026
import pandas as pd

# Load the first sheet into a DataFrame
df_rural = pd.read_excel('./data/raw/cba_historicos/Historico-por-grupo-alimenticio-y-por-producto-CBAR-2024-2026-2-1.xlsx', sheet_name='Histórico producto CBAR', index_col=False)
df_rural.head()


,Año,Mes,Producto,Cantidad base,Unidad de medida base,Kilocalorias diarias,Kilocalorias mensuales,Cantidad gramos diarios,Cantidad gramos mensuales,Costo diario,Costo mensual,Precio según unidad de medida base*
0,2024,Enero,Arroz corriente,454.0,Gramos,61.358637,1840.759113,17.044066,511.321976,0.298271,8.948135,7.945000
1,2024,Enero,Arroz precocido,454.0,Gramos,11.877317,356.319524,3.125610,93.768296,0.057824,1.734713,8.399000
2,2024,Enero,Maíz blanco,454.0,Gramos,287.996160,8639.884810,78.903058,2367.091729,0.434488,13.034646,2.500000
3,2024,Enero,Harina de maíz,454.0,Gramos,44.918127,1347.543820,12.442695,373.280837,0.171087,5.132612,6.242500
4,2024,Enero,Harina para atoles,454.0,Gramos,17.359381,520.781424,4.653989,139.619685,0.113764,3.412926,11.097778


In [30]:
df_rural.columns

Index(['Año', 'Mes', 'Producto', 'Cantidad base', 'Unidad de medida base',
       'Kilocalorias diarias', 'Kilocalorias mensuales',
       'Cantidad gramos diarios', 'Cantidad gramos mensuales', 'Costo diario',
       'Costo mensual', 'Precio según unidad de medida base*'],
      dtype='str')

In [31]:
rows_with_nan = df_rural[df_rural.isna().any(axis=1)]

rows_with_nan

,Año,Mes,Producto,Cantidad base,Unidad de medida base,Kilocalorias diarias,Kilocalorias mensuales,Cantidad gramos diarios,Cantidad gramos mensuales,Costo diario,Costo mensual,Precio según unidad de medida base*
1680,*Precio según la unidad de medida base estable...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [32]:
df_rural.shape

(1681, 12)

In [33]:
df_rural = df_rural.dropna()
df_rural.shape

(1680, 12)

In [34]:
from dtos.Product import Product, Products

products = Products(regions=['urbana', 'republica', 'rural'])
current_region = 'rural'
for index, row in df_rural.iterrows(): 
    result = incap_data.search_word_matches(str(row["Producto"]).lower(), target_field='nombre', top_n=1, min_probability=.5)
    energy_per_100g_kcal = 100 * row["Kilocalorias diarias"] / row["Cantidad gramos diarios"]
    cost_per_gram = row["Costo diario"] / row["Cantidad gramos diarios"]
    days_in_month = row['Cantidad gramos mensuales']/ row["Cantidad gramos diarios"]
    
    
    products.add_data(
        current_region, 
        row["Producto"], 
        energy_per_100g_kcal, 
        row["Año"],
        row["Mes"],
        cost_per_gram,
        row['Kilocalorias diarias'],
        row['Cantidad gramos diarios'],
        days_in_month,
        row['Cantidad base'],
        row['Costo diario'],
        result[0][0] if result else None, 
        "Search Word matches with min prob 50%" if result else None,
        result[0][1] if result else None, 
    )

Nuevo producto creado para Arroz corriente ; 2024, Enero
Nuevo producto creado para Arroz precocido ; 2024, Enero
Nuevo producto creado para Maíz blanco ; 2024, Enero
Nuevo producto creado para Harina de maíz ; 2024, Enero
Nuevo producto creado para Harina para atoles  ; 2024, Enero
Nuevo producto creado para Pan francés ; 2024, Enero
Nuevo producto creado para Pan dulce ; 2024, Enero
Nuevo producto creado para Galletas dulces ; 2024, Enero
Nuevo producto creado para Tortillas frescas ; 2024, Enero
Nuevo producto creado para Avena/mosh ; 2024, Enero
Nuevo producto creado para Espagueti ; 2024, Enero
Nuevo producto creado para Fideos en todas sus formas (Excepto macarrones y espagueti) ; 2024, Enero
Nuevo producto creado para Carne de res sin hueso (posta) ; 2024, Enero
Nuevo producto creado para Carne de res con hueso para cocido ; 2024, Enero
Nuevo producto creado para Carne de pollo blanco ; 2024, Enero
Nuevo producto creado para Carne de pollo amarillo  ; 2024, Enero
Nuevo producto 

In [35]:
df_urbano = pd.read_excel('./data/raw/cba_historicos/Historico-por-grupo-alimenticio-y-por-producto-CBAU-2024-2026.xlsx', sheet_name='Histórico producto CBAU', index_col=False)
df_urbano.head()

,Año,Mes,Producto,Cantidad base,Unidad de medida base,Kilocalorías diarias,Kilocalorías mensuales,Cantidad gramos diarios,Cantidad gramos mensuales,Costo diario,Costo mensual,Precio según unidad de medida base*
0,2024,Enero,Arroz corriente,454.0,Gramos,66.715985,2001.479553,18.532218,555.966542,0.324314,9.729414,7.945000
1,2024,Enero,Arroz precocido,454.0,Gramos,22.246412,667.392371,5.854319,175.629571,0.108305,3.249147,8.399000
2,2024,Enero,Pan francés,454.0,Gramos,150.071298,4502.138948,54.177364,1625.320920,1.135001,34.050019,9.511173
3,2024,Enero,Pan dulce,454.0,Gramos,135.949815,4078.494449,37.043546,1111.306389,0.700037,21.001118,8.579549
4,2024,Enero,Galletas dulces,454.0,Gramos,6.280560,188.416796,1.330627,39.918813,0.065367,1.961012,22.302750


In [36]:
rows_with_nan = df_urbano[df_urbano.isna().any(axis=1)]

rows_with_nan

,Año,Mes,Producto,Cantidad base,Unidad de medida base,Kilocalorías diarias,Kilocalorías mensuales,Cantidad gramos diarios,Cantidad gramos mensuales,Costo diario,Costo mensual,Precio según unidad de medida base*
1848,*Precio según la unidad de medida base estable...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [37]:
df_urbano.shape

(1849, 12)

In [38]:
df_urbano = df_urbano.dropna()
df_urbano.shape

(1848, 12)

In [39]:
current_region = 'urbana'
for index, row in df_urbano.iterrows(): 
    result = incap_data.search_word_matches(str(row["Producto"]).lower(), target_field='nombre', top_n=1, min_probability=.5)
    energy_per_100g_kcal = 100 * row["Kilocalorías diarias"] / row["Cantidad gramos diarios"]
    cost_per_gram = row["Costo diario"] / row["Cantidad gramos diarios"]
    days_in_month = row['Cantidad gramos mensuales']/ row["Cantidad gramos diarios"]

    
    products.add_data(
        current_region, 
        row["Producto"], 
        energy_per_100g_kcal, 
        row["Año"],
        row["Mes"],
        cost_per_gram,
        row['Kilocalorías diarias'],
        row['Cantidad gramos diarios'],
        days_in_month,
        row['Cantidad base'],
        row['Costo diario'],
        result[0][0] if result else None, 
        "Search Word matches with min prob 50%" if result else None,
        result[0][1] if result else None, 
    )

Nuevo producto creado para Arroz corriente ; 2024, Enero
Nuevo producto creado para Arroz precocido ; 2024, Enero
Nuevo producto creado para Pan francés ; 2024, Enero
Nuevo producto creado para Pan dulce ; 2024, Enero
Nuevo producto creado para Galletas dulces ; 2024, Enero
Nuevo producto creado para Tortillas frescas ; 2024, Enero
Nuevo producto creado para Cereales de bolsa o caja ; 2024, Enero
Nuevo producto creado para Espagueti ; 2024, Enero
Nuevo producto creado para Fideos en todas sus formas (Excepto macarrones y espagueti) ; 2024, Enero
Nuevo producto creado para Carne de res molida ; 2024, Enero
Nuevo producto creado para Carne de res para asar (con y sin hueso) ; 2024, Enero
Nuevo producto creado para Carne de res sin hueso (posta) ; 2024, Enero
Nuevo producto creado para Carne de res con hueso para cocido ; 2024, Enero
Nuevo producto creado para Posta de cerdo sin hueso  ; 2024, Enero
Nuevo producto creado para Carne de pollo blanco ; 2024, Enero
Nuevo producto creado para 

In [40]:
products